# 03 — Probability, likelihood, softmax

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/003-probability-expectation-likelihood.ipynb)

A model that outputs probabilities is judged by how much probability it gave to what actually happened. In this notebook you score ten coin flips, plot the cost of each guess, and drag a slider to see softmax turn three raw scores into a confident or a flat prediction.

## 1. The numbers

Seven heads out of ten flips is the data from notebook 004, where y = 10 and ŷ = 7. The guess p = 7/10 is the frequency of heads; the likelihood multiplies the chance of every flip, and the negative log turns that product into a number you can minimise.

In [ ]:
import numpy as np                             # NumPy is preinstalled on Colab
heads = 7                                     # successes observed: the 7 from notebook 004
flips = 10                                    # total tries: the 10 from notebook 004
p = heads / flips                             # the natural guess: the observed frequency, 0.7
like = p ** heads * (1 - p) ** (flips - heads)   # multiply the chance of each of the ten flips
nll = -np.log(like)                           # negative log-likelihood: lower is better
print("p =", p)                               # 0.7
print("likelihood =", like)                   # about 0.00222
print("NLL =", nll)                           # about 6.108

**Line by line**

- `import numpy as np` — loads NumPy, which provides `log`.
- `heads = 7` — the number of heads, matching the 7 in notebook 004.
- `flips = 10` — the number of tosses, matching the 10 in notebook 004.
- `p = heads / flips` — the maximum-likelihood guess for a coin, 0.7.
- `like = ...` — likelihood: p for each head, 1 − p for each tail, all multiplied.
- `nll = -np.log(like)` — negative log-likelihood, the loss we minimise.
- `print("p =" ...)` — the guessed probability of heads.
- `print("likelihood =" ...)` — the tiny product itself.
- `print("NLL =" ...)` — the same information as one manageable number.

## 2. Step by step

Likelihood is a product of ten small numbers, which underflows fast. Adding the surprises −log p instead gives the same total without the tiny product, so this is how libraries compute it.

In [ ]:
import numpy as np                             # for the logarithm
outcomes = [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]      # seven heads (1) followed by three tails (0)
p = 0.7                                        # the guessed probability of heads
total = 0.0                                    # running negative log-likelihood

for k, o in enumerate(outcomes, 1):            # visit each flip, numbering from 1
    q = p if o == 1 else 1 - p                 # probability the guess gave to this outcome
    surprise = -np.log(q)                      # surprisal: rare outcomes cost more
    total = total + surprise                   # add this flip's cost to the running total
    print(f"flip {k}: outcome {o}, p = {q:.2f}, -log p = {surprise:.3f}")   # one row per flip

print("NLL =", total)                          # about 6.108, the same as section 1

**Line by line**

- `import numpy as np` — needed for `np.log`.
- `outcomes = [...]` — the ten flips in order: seven 1s, then three 0s.
- `p = 0.7` — the probability the model assigns to heads.
- `total = 0.0` — accumulator for the NLL.
- `for k, o in enumerate(outcomes, 1)` — `k` is the flip number, `o` the outcome.
- `q = p if o == 1 else 1 - p` — probability of the outcome that actually occurred.
- `surprise = -np.log(q)` — the cost of that one outcome.
- `total = total + surprise` — sum the costs; a sum of logs is a log of products.
- `print(f"flip ...")` — one line per flip showing `q` and its cost.
- `print("NLL =" ...)` — the total, matching section 1.

## 3. Plot: NLL against the guessed probability

Sweep the guess from 0.01 to 0.99 and the cost forms a valley. Its bottom sits at the observed frequency, 0.7, which is why counting heads is the right answer for a coin.

In [ ]:
import numpy as np                             # for the log and the grid
import matplotlib.pyplot as plt                # plotting library on Colab

ps = np.linspace(0.01, 0.99, 99)               # 99 guesses between 0.01 and 0.99
curve = -(7 * np.log(ps) + 3 * np.log(1 - ps)) # NLL for 7 heads and 3 tails at each guess

plt.figure(figsize=(6, 3))                     # a wide, short figure
plt.plot(ps, curve, color="tab:blue")          # the cost curve
plt.axvline(0.7, color="tab:red", linestyle="--", label="p = 0.7")   # where the valley bottoms out
plt.xlabel("p (guessed probability of heads)") # x-axis: the guess
plt.ylabel("negative log-likelihood")          # y-axis: the cost
plt.title("NLL is lowest at p = 0.7, the observed frequency")   # takeaway
plt.legend()                                   # name the dashed line
plt.show()                                     # render the figure

**Line by line**

- `import numpy as np` — for `linspace` and `log`.
- `import matplotlib.pyplot as plt` — for the plot.
- `ps = np.linspace(0.01, 0.99, 99)` — a grid of candidate probabilities, avoiding 0 and 1.
- `curve = ...` — NLL at every guess: 7 heads and 3 tails.
- `plt.figure(...)` — create the canvas.
- `plt.plot(ps, curve, ...)` — draw the valley.
- `plt.axvline(0.7, ...)` — mark the guess that matches the data.
- `plt.xlabel(...)` / `plt.ylabel(...)` — guess on x, cost on y.
- `plt.title(...)` — the valley's minimum in words.
- `plt.legend()` / `plt.show()` — label the marker and render.

## 4. Interactive: softmax, sharpened by a slider

A classifier does not output probabilities directly; it outputs raw scores called logits. Softmax exponentiates each score and divides by the total, so the three numbers become positive and sum to 1. The slider multiplies the logits first: a big scale makes the winner dominate, a small scale flattens all three.

In [ ]:
from ipywidgets import interact                # sliders that rerun a function
import numpy as np                             # exp and sums
import matplotlib.pyplot as plt                # bars for each class

logits = np.array([2.0, 1.0, 0.0])             # three raw scores, one per class

def softmax(z):                                # turn scores into probabilities
    e = np.exp(z - z.max())                    # subtract the max so exp cannot overflow
    return e / e.sum()                         # divide by the total so the parts sum to 1

print("scale 1:", np.round(softmax(logits), 3))   # [0.665 0.245 0.09]

def show(scale=1.0):                           # scale comes from the slider
    p = softmax(scale * logits)                # sharpen (large scale) or flatten (small scale)
    plt.figure(figsize=(5, 3))                 # a fresh figure per drag
    plt.bar(["class 0", "class 1", "class 2"], p, color="tab:blue")   # one bar per class
    plt.ylim(0, 1)                             # fixed axis so the bars do not jump
    plt.xlabel("class")                        # x-axis label
    plt.ylabel("probability")                  # y-axis label
    plt.title(f"scale = {scale:.1f}: top p = {p.max():.3f}, sum = {p.sum():.3f}")   # live readout
    plt.show()                                 # draw the frame

interact(show, scale=(0.1, 10.0, 0.1))         # slider over the logit scale

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `import numpy as np` — for `exp`, `max` and `sum`.
- `import matplotlib.pyplot as plt` — for the bar chart.
- `logits = np.array([2.0, 1.0, 0.0])` — the three raw class scores.
- `def softmax(z)` — the function that normalises scores into probabilities.
- `e = np.exp(z - z.max())` — exponentiate; subtracting the max avoids overflow.
- `return e / e.sum()` — divide by the total; the result sums to 1.
- `print("scale 1:" ...)` — the base case, about [0.665, 0.245, 0.090].
- `def show(scale=1.0)` — the function that redraws one frame.
- `p = softmax(scale * logits)` — multiply the logits, then normalise.
- `plt.figure(...)` / `plt.bar(...)` — three bars, one per class.
- `plt.ylim(0, 1)` — hold the axis fixed so changes are comparable.
- `plt.xlabel(...)` / `plt.ylabel(...)` / `plt.title(...)` — labels and the live readout.
- `plt.show()` — render the frame.
- `interact(show, scale=(0.1, 10.0, 0.1))` — the slider, its range and step.

## 5. When it breaks: confident and wrong

The NLL of the correct class is −log(1 − q), where q is the probability given to the wrong class. As q approaches 1 that cost shoots to infinity. Notebook 004's squared error was 9; here a 99%-confident mistake already costs 4.6, and 99.9% costs 6.9.

In [ ]:
import numpy as np                             # for log and linspace
import matplotlib.pyplot as plt                # plotting library

q = np.linspace(0.001, 0.999, 200)             # probability given to the wrong class
nll = -np.log(1 - q)                           # cost of the right class, which got only 1 - q

plt.figure(figsize=(6, 3))                     # create the figure
plt.plot(q, nll, color="tab:red")              # the cost curve
plt.yscale("log")                              # the last sliver of probability explodes
plt.axvline(0.99, color="grey", linestyle="--")   # a very confident mistake
plt.xlabel("probability assigned to the wrong class")   # x-axis: misplaced confidence
plt.ylabel("NLL of the correct class")         # y-axis: the cost
plt.title("99% confident and wrong costs 4.6; 99.9% costs 6.9")   # takeaway
plt.show()                                     # render the figure

print("q = 0.99 -> NLL =", round(-np.log(0.01), 3))    # 4.605
print("q = 0.999 -> NLL =", round(-np.log(0.001), 3))  # 6.908

**Line by line**

- `import numpy as np` — for `linspace` and `log`.
- `import matplotlib.pyplot as plt` — for the plot.
- `q = np.linspace(0.001, 0.999, 200)` — confidence on the wrong class, from 0.1% to 99.9%.
- `nll = -np.log(1 - q)` — cost of the correct class's leftover probability.
- `plt.figure(...)` — create the canvas.
- `plt.plot(q, nll, ...)` — draw the exploding cost.
- `plt.yscale("log")` — a log axis makes the explosion readable.
- `plt.axvline(0.99, ...)` — mark 99% confidence.
- `plt.xlabel(...)` / `plt.ylabel(...)` — confidence on x, cost on y.
- `plt.title(...)` — the two costs in words.
- `plt.show()` — render the figure.
- `print("q = 0.99 ...")` — the exact cost of a 99% mistake.
- `print("q = 0.999 ...")` — the exact cost of a 99.9% mistake.

## 6. Compare: sharpening helps when right, hurts when wrong

Take the same logits, [2, 1, 0], and sharpen them by a factor of 1, 2 and 3. If class 0 is the true label the NLL falls; if class 1 is the true label the same sharpening raises it.

In [ ]:
import numpy as np                             # for exp and log
import matplotlib.pyplot as plt                # plotting library

scales = [1.0, 2.0, 3.0]                       # three amounts of sharpening
right, wrong = [], []                          # NLL when class 0 / class 1 is true

for s in scales:                               # one group per scale
    z = s * np.array([2.0, 1.0, 0.0])          # the same logits, multiplied by the scale
    e = np.exp(z - z.max())                    # exponentiate, shifted for safety
    p = e / e.sum()                            # softmax probabilities
    right.append(-np.log(p[0]))                # cost if class 0 was the answer
    wrong.append(-np.log(p[1]))                # cost if class 1 was the answer

x = np.arange(len(scales))                     # bar positions: 0, 1, 2
plt.figure(figsize=(6, 3))                     # create the figure
plt.bar(x - 0.2, right, width=0.4, label="true class = 0", color="tab:green")   # when right
plt.bar(x + 0.2, wrong, width=0.4, label="true class = 1", color="tab:red")     # when wrong
plt.xticks(x, [str(s) for s in scales])        # label each group with its scale
plt.xlabel("logit scale")                      # x-axis label
plt.ylabel("NLL of the true class")            # y-axis label
plt.title("sharpening: 0.41 -> 0.05 when right, 1.41 -> 3.05 when wrong")   # takeaway
plt.legend()                                   # name the two bars
plt.show()                                     # render the figure

**Line by line**

- `import numpy as np` — for `exp`, `log` and `arange`.
- `import matplotlib.pyplot as plt` — for the grouped bars.
- `scales = [1.0, 2.0, 3.0]` — the sharpening factors being compared.
- `right, wrong = [], []` — two lists: cost when class 0 or class 1 is true.
- `for s in scales` — one pair of bars per scale.
- `z = s * np.array([2.0, 1.0, 0.0])` — sharpen the logits.
- `e = np.exp(z - z.max())` — exponentiate safely.
- `p = e / e.sum()` — the softmax probabilities.
- `right.append(-np.log(p[0]))` — NLL if class 0 is correct.
- `wrong.append(-np.log(p[1]))` — NLL if class 1 is correct.
- `x = np.arange(len(scales))` — bar positions for the three groups.
- `plt.bar(x - 0.2, ...)` / `plt.bar(x + 0.2, ...)` — the green and red bars side by side.
- `plt.xticks(...)` — print the scale under each group.
- `plt.xlabel(...)` / `plt.ylabel(...)` / `plt.title(...)` — labels and the comparison.
- `plt.legend()` / `plt.show()` — name the bars and render.

## Your turn

Drag the scale slider to 5 and predict the top probability before you read the title. Then set the scale back to 1 and change the logits to `[0, 2, 1]` — which class wins, and does the sum still equal 1?